# working version of the transformer version of the chatbot

In [1]:
!pip install sentence-transformers faiss-cpu transformers accelerate bitsandbytes nltk bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18

In [21]:
import os

corpus = []
folder_path = "/content/Research-Chatbot"  # Adjust this to your folder

for file_name in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file_name)
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            corpus.extend([para.strip() for para in content.split('\n') if len(para.strip()) > 50])
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

print(f"Loaded {len(corpus)} paragraphs.")


Error reading /content/Research-Chatbot/.ipynb_checkpoints: [Errno 21] Is a directory: '/content/Research-Chatbot/.ipynb_checkpoints'
Loaded 3221 paragraphs.


In [22]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = model.encode(corpus, show_progress_bar=True, convert_to_numpy=True)


Batches:   0%|          | 0/101 [00:00<?, ?it/s]

In [23]:
import faiss
import numpy as np

embedding_dim = corpus_embeddings.shape[1]  # Should be 384 for all-MiniLM-L6-v2
index = faiss.IndexFlatL2(embedding_dim)
index.add(corpus_embeddings)

print(f"FAISS index built with {index.ntotal} vectors.")


FAISS index built with 3221 vectors.


In [24]:
def retrieve_passages(query, top_k=3):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)
    results = [corpus[idx] for idx in indices[0]]
    return results


In [25]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

t5_model = T5ForConditionalGeneration.from_pretrained("t5-base")
t5_tokenizer = T5Tokenizer.from_pretrained("t5-base")

def generate_answer(query, context_passages):
    context = " ".join(context_passages)
    prompt = f"question: {query} context: {context}"
    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, padding=True)
    outputs = t5_model.generate(**inputs, max_length=128)
    answer = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer


In [26]:
query = "What are library opening hours?"
passages = retrieve_passages(query)

print("\nTop Retrieved Passages:")
for i, p in enumerate(passages):
    print(f"[{i+1}] {p}\n")

answer = generate_answer(query, passages)
print("Generated Answer:\n", answer)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



Top Retrieved Passages:
[1] What hours is the library open?                                                                                                                            10

[2] As the days grow longerour opening hours get shorter. We are now open Monday to Saturday9am to 5pm.See all the opening hours here!

[3] As the days grow longerour opening hours get shorter. We are now open Monday to Saturday9am to 5pm.See all the opening hours here!

Generated Answer:
 Monday to Saturday9am to 5pm


In [27]:
def start_chatbot(top_k=3):
    print("🎓 College Brochure Chatbot is ready! Type 'exit' to quit.\n")

    while True:
        query = input("You: ")
        if query.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break

        retrieved_passages = retrieve_passages(query)
        answer = generate_answer(query,retrieved_passages)

        # Step 5: Show the answer
        print("🤖 Chatbot:", answer)
        print("-" * 60)

In [28]:
#start_chatbot()


In [29]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Load the TinyLLaMA Chat model
model_id_tinyllm = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_tinyllm = AutoTokenizer.from_pretrained(model_id_tinyllm)
model_tinyllm = AutoModelForCausalLM.from_pretrained(model_id_tinyllm, device_map="auto", torch_dtype="auto")

In [30]:
def refine_answer_llama(query, raw_answer):
    prompt = f"""<|system|>You are an intelligent chatbot. Your role is to convert the retreived chunks into humanized text..<|end|>
<|user|>Question: {query}
Answer: {raw_answer}
Convert the following text to a chatbot response. Add greetings and ask the user if they have any further questions. AT the end of the conversion, say Thank You.<|end|>
<|assistant|>"""

    inputs = tokenizer_tinyllm(prompt, return_tensors="pt").to(model_tinyllm.device)
    outputs = model_tinyllm.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer_tinyllm.eos_token_id
    )

    full_output = tokenizer_tinyllm.decode(outputs[0], skip_special_tokens=True)
    # Extract the assistant's response
    if "<|assistant|>" in full_output:
        return full_output.split("<|assistant|>")[-1].strip()
    return full_output.strip()


In [31]:
query = "How many books does the DBS Library have?"
raw_answer = "over 43,000"

#refined = refine_answer_llama(query, raw_answer)
#print("📘 Refined Answer:", refined)

In [32]:
def start_chatbot_v2(top_k=3):
    print("🎓 College Brochure Chatbot is ready! Type 'exit' to quit.\n")

    while True:
        query = input("You: ")
        if query.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break

        retrieved_passages = retrieve_passages(query)
        answer = generate_answer(query,retrieved_passages)
        print("🤖 Chatbot original answer:", answer)
        print("-" * 60)

        refined_answer = refine_answer_llama(query, answer)

        # Step 5: Show the answer
        print("🤖 Chatbot refined answer:", refined_answer)
        print("-" * 60)

In [33]:
#start_chatbot_v2()

Evaluation Pipeline

In [34]:
import nltk
nltk.download('punkt')

import time
from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [35]:
test_set = [
    {"query": "How many books DBS Library has?", "expected_answer": "over 43,000"},
     {"query": "Where can I access DBS WiFi?", "expected_answer": ""},
      {"query": "What are library opening hours?", "expected_answer": " 24 hours a day"},
       {"query": "What ratings did DBS earned?", "expected_answer": " 4 stars"},
        {"query": "how to view Library account?", "expected_answer": ""},
         {"query": "Are the Guides to Library resources for students with disabilities are also available in the Library?", "expected_answer": " on the library website"},
          {"query": "How many university partnerships does DBS has developed?", "expected_answer": " over 75"}
  ]

  #create a program to load this from file





In [40]:
file_path = r"/content/Research-Chatbot/Test_set_v5.0.json"
import json

# Load test_set from file
def load_test_set(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"The file {path} does not exist.")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

if __name__ == "__main__":
    try:
        test_set = load_test_set(file_path)
        print("Test set loaded successfully.")
        for item in test_set:
            print(f"Query: {item.get('query', '')}")
            print(f"Expected Answer: {item.get('expected_answer', '')}")
            print("-" * 50)
    except Exception as e:
        print(f"Error: {e}")


Test set loaded successfully.
Query: What is PSI?
Expected Answer: professional body for psychology in Ireland
--------------------------------------------------
Query: What country has the highest reputation for leading independent colleges?
Expected Answer: Ireland
--------------------------------------------------
Query: student population of DBS?
Expected Answer: over 9,000
--------------------------------------------------
Query: What is the name of Ireland's largest independent third-level college?
Expected Answer: Dublin Business School
--------------------------------------------------
Query: When was Dublin Business School established?
Expected Answer: 1975
--------------------------------------------------
Query: When was DBS established?
Expected Answer: 1975
--------------------------------------------------
Query: DBS was established in which year?
Expected Answer: 1975
--------------------------------------------------
Query: What is the name of the society that represent

In [41]:
#test_set

In [42]:


smoothie = SmoothingFunction().method4

def evaluate_rag_model(test_set):
    results = []
    total_time = 0
    all_generated = []
    all_expected = []

    for i, item in enumerate(test_set):
        query = item["query"]
        expected = item["expected_answer"]
        print(f"Evaluating Query {i+1}/{len(test_set)}")
        start_time = time.time()
        retrieved_passages = retrieve_passages(query)
        generated = generate_answer(query,retrieved_passages)
        end_time = time.time()

        response_time = end_time - start_time
        total_time += response_time

        # Save for BERTScore later
        all_generated.append(generated)
        all_expected.append(expected)

        # Exact match
        exact_match = int(expected.lower() in generated.lower())

        # BLEU Score
        reference = [expected.split()]
        candidate = generated.split()
        bleu = sentence_bleu(reference, candidate, smoothing_function=smoothie)

        results.append({
            "Query": query,
            "Generated": generated,
            "Expected": expected,
            "ExactMatch": exact_match,
            "BLEU": bleu,
            "TimeTaken": response_time
        })

    # BERTScore
    P, R, F1 = bert_score(all_generated, all_expected, lang="en", verbose=True)
    avg_bertscore_f1 = F1.mean().item()

    # Summary metrics
    accuracy = sum(r["ExactMatch"] for r in results) / len(results)
    avg_bleu = sum(r["BLEU"] for r in results) / len(results)
    avg_time = total_time / len(results)

    print(f"\n--- Evaluation Summary ---")
    print(f"Accuracy (Exact Match): {accuracy:.2f}")
    print(f"Average BLEU Score: {avg_bleu:.2f}")
    print(f"Average BERTScore F1: {avg_bertscore_f1:.2f}")
    print(f"Average Inference Time: {avg_time:.2f} seconds\n")

    return results


In [43]:
results = evaluate_rag_model(test_set)
results

Evaluating Query 1/172
Evaluating Query 2/172
Evaluating Query 3/172
Evaluating Query 4/172
Evaluating Query 5/172
Evaluating Query 6/172
Evaluating Query 7/172
Evaluating Query 8/172
Evaluating Query 9/172
Evaluating Query 10/172
Evaluating Query 11/172
Evaluating Query 12/172
Evaluating Query 13/172
Evaluating Query 14/172
Evaluating Query 15/172
Evaluating Query 16/172
Evaluating Query 17/172
Evaluating Query 18/172
Evaluating Query 19/172
Evaluating Query 20/172
Evaluating Query 21/172
Evaluating Query 22/172
Evaluating Query 23/172
Evaluating Query 24/172
Evaluating Query 25/172
Evaluating Query 26/172
Evaluating Query 27/172
Evaluating Query 28/172
Evaluating Query 29/172
Evaluating Query 30/172
Evaluating Query 31/172
Evaluating Query 32/172
Evaluating Query 33/172
Evaluating Query 34/172
Evaluating Query 35/172
Evaluating Query 36/172
Evaluating Query 37/172
Evaluating Query 38/172
Evaluating Query 39/172
Evaluating Query 40/172
Evaluating Query 41/172
Evaluating Query 42/172
E

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 41.29 seconds, 4.17 sentences/sec

--- Evaluation Summary ---
Accuracy (Exact Match): 0.84
Average BLEU Score: 0.68
Average BERTScore F1: 0.96
Average Inference Time: 1.44 seconds



[{'Query': 'What is PSI?',
  'Generated': 'to promote and support psychology education and practice in Ireland Higher Diploma in Arts in Psychology',
  'Expected': 'professional body for psychology in Ireland',
  'ExactMatch': 0,
  'BLEU': 0.04030833724870907,
  'TimeTaken': 1.9852635860443115},
 {'Query': 'What country has the highest reputation for leading independent colleges?',
  'Generated': 'Ireland',
  'Expected': 'Ireland',
  'ExactMatch': 1,
  'BLEU': 1.0,
  'TimeTaken': 0.5663714408874512},
 {'Query': 'student population of DBS?',
  'Generated': 'over 9,000',
  'Expected': 'over 9,000',
  'ExactMatch': 1,
  'BLEU': 0.2213885886251307,
  'TimeTaken': 0.8200027942657471},
 {'Query': "What is the name of Ireland's largest independent third-level college?",
  'Generated': "Ireland's Leading Independent College",
  'Expected': 'Dublin Business School',
  'ExactMatch': 0,
  'BLEU': 0,
  'TimeTaken': 0.7415142059326172},
 {'Query': 'When was Dublin Business School established?',
  '